# Test GPT Model
Fill in one TODO in `src/model.py`, then run the corresponding cell to verify it works.

In [2]:
import sys
sys.path.extend(["..", "../src"])   # add project root and src/ to path

import torch
from transformers import AutoTokenizer

from config import GPTConfig
from model import GPT, count_parameters, CausalSelfAttention, MLP, TransformerBlock

cfg = GPTConfig()
print(f"d_model={cfg.d_model}, n_layers={cfg.n_layers}, n_heads={cfg.n_heads}")

d_model=256, n_layers=6, n_heads=8


---
## Test 1: MLP

In [3]:
# fill in MLP first, then run this
mlp = MLP(cfg)
x = torch.randn(2, 8, cfg.d_model)
y = mlp(x)
print(f"MLP forward OK: {list(x.shape)} -> {list(y.shape)}")
assert y.shape == x.shape, f"shape mismatch {y.shape} != {x.shape}"

MLP forward OK: [2, 8, 256] -> [2, 8, 256]


---
## Test 2: CausalSelfAttention

In [4]:
# fill in attention forward, then run this
attn = CausalSelfAttention(cfg)
x = torch.randn(2, 8, cfg.d_model)
y = attn(x)
print(f"Attention forward OK: {list(x.shape)} -> {list(y.shape)}")
assert y.shape == x.shape, f"shape mismatch {y.shape} != {x.shape}"
print("Causal mask works: all good")

Attention forward OK: [2, 8, 256] -> [2, 8, 256]
Causal mask works: all good


---
## Test 3: TransformerBlock

In [5]:
# fill in TransformerBlock, then run this
block = TransformerBlock(cfg)
x = torch.randn(2, 8, cfg.d_model)
y = block(x)
print(f"Block forward OK: {list(x.shape)} -> {list(y.shape)}")
assert y.shape == x.shape, f"shape mismatch {y.shape} != {x.shape}"

Block forward OK: [2, 8, 256] -> [2, 8, 256]


---
## Test 4: Full GPT (forward + loss)

In [7]:
# fill in GPT __init__ and forward, then run this
model = GPT(cfg)
n_params = count_parameters(model)
print(f"Total params: {n_params:.2f}M")

# forward with loss
x = torch.randint(0, cfg.vocab_size, (2, 64))
logits, loss = model(x, targets=x)
print(f"Forward OK: logits {list(logits.shape)}, loss {loss.item():.4f}")
assert logits.shape == (2, 64, cfg.vocab_size), f"logits shape wrong: {logits.shape}"
assert loss.item() > 0, "loss should be positive"

Total params: 30.59M
Forward OK: logits [2, 64, 50257], loss 10.8823


---
## Test 5: Generation

In [8]:
# Test weight decay grouping — verify _configure_optimizers works
from config import TrainConfig
from pretrain import Trainer

model = GPT(cfg)
trainer = Trainer(model, TrainConfig(), 'cpu')

# Check groups
for i, group in enumerate(trainer.optimizer.param_groups):
    print(f"Group {i}: weight_decay={group['weight_decay']}, {len(group['params'])} params")
    # Print first 3 param names to verify
    count = 0
    for p in group['params']:
        for name, param in model.named_parameters():
            if param.data_ptr() == p.data_ptr():
                print(f"  {name}")
                count += 1
                if count >= 3:
                    break
        if count >= 3:
            break
    print()

# Verify total params match
decay_count = sum(p.numel() for g in trainer.optimizer.param_groups for p in g['params'] if g['weight_decay'] > 0)
no_decay_count = sum(p.numel() for g in trainer.optimizer.param_groups for p in g['params'] if g['weight_decay'] == 0)
total = decay_count + no_decay_count
print(f"Decay params: {decay_count:,}")
print(f"No-decay params: {no_decay_count:,}")
print(f"Total: {total:,} (expected ~{count_parameters(model) * 1e6:.0f})")
assert total == sum(p.numel() for p in model.parameters()), "param count mismatch!"
print("✅ Weight decay grouping OK")

Group 0: weight_decay=0.1, 25 params
  blocks.0.attn.c_attn.weight
  blocks.0.attn.c_proj.weight
  blocks.0.mlp.fc.0.weight

Group 1: weight_decay=0.0, 53 params
  blocks.0.attn.c_attn.bias
  blocks.0.attn.c_proj.bias
  blocks.0.ln1.bias

Decay params: 17,584,384
No-decay params: 13,002,065
Total: 30,586,449 (expected ~30586449)
✅ Weight decay grouping OK


d:\宸铭\学习\t大课程\AI4S交叉实践-深度学习\deep-learning\hw4\notebooks\../src\pretrain.py:79: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.scaler = GradScaler(enabled=train_cfg.use_amp)


---
## Test 6: Weight Decay Grouping (TODO2)

Ensure the optimizer correctly separates decay/no-decay params.

In [9]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

prompt = tokenizer("Once upon a time", return_tensors="pt")["input_ids"]
output = model.generate(prompt, max_new_tokens=30, temperature=1.0)
generated = tokenizer.decode(output[0])
print("Generated:")
print(generated)

Generated:
Once upon a timedepending vegan dogSportundaebtedDButherford renewable reckikes answered366oub Napoleonibe FY steroidsNYSE Ethertenessovicrop Target letting defective furiouslyident Hale Parker
